[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hankpark0706/OL7014/blob/main/notebooks/week6_lap.ipynb)

In [ ]:
%pip install -q gurobipy

In [ ]:
import gurobipy as gp
from gurobipy import GRB

params = {
    "WLSACCESSID": "paste your ACCESSID here",
    "WLSSECRET":   "paste your SECRET here",
    "LICENSEID":   123456,   # the number itself, no quotes
}
env = gp.Env(params=params)

# LAP --- the game's five nodes

![the network](https://raw.githubusercontent.com/hankpark0706/OL7014/main/notebooks/img/week6_lap_network.png)

$$
\begin{aligned}
\min \quad & 100 \textstyle\sum_{j} x_j \;+\; 10 \sum_{j} r_j \;+\; \sum_{(i,j)} t_{ij}\, y_{ij} \;+\; 60 \sum_{j} q_j \;+\; \sum_{j} s_j \\
\text{s.t.} \quad & \textstyle\sum_{j} y_{ij} - \sum_{j} y_{ji} = r_i - d_i + q_i - s_i \quad (\forall i) \\
& \textstyle\sum_{j} r_j \le 55, \qquad r_j \le 55\, x_j, \qquad \sum_{j} x_j \le 2 \\
& y, r, s \ge 0, \quad 0 \le q_j \le d_j, \quad x_j \in \{0,1\}
\end{aligned}
$$

Each road is two directed arcs: $y_{AC}$ and $y_{CA}$ both exist, same cost.

**Math to code**

| math | code |
| :--- | :--- |
| $x_j \in \{0,1\}$ | `addVar(vtype=GRB.BINARY)` |
| $y, r, s \ge 0$ | `addVar()` --- continuous, lower bound `0`, is the default |
| $0 \le q_A \le 12$ | `addVar(ub=12)` |
| each row | one `addConstr` --- keep a handle (`nodeE = ...`), we edit it later |

## Live coding --- fill in the blanks

Skeleton for building the model live in class, one comment at a time.

In [ ]:
# Initialize the model gp.Model()


# switches: xA, ..., xE binary --- every node is a candidate site


# reserves rA, ..., rE and the ten arc flows yAC, yCA, yBC, yCB, yCD, yDC, yCE, yEC, yDE, yED


# shortfalls qA (ub=12), ..., qE (ub=24) and surpluses sA, ..., sE


# objective: open + hold + ship + fail + waste --- lap.setObjective(..., GRB.MINIMIZE)


# conservation, one row per node --- keep handles: nodeA = lap.addConstr(...)


# budget, linking, at most p = 2 facilities


# Model is set up. Let's optimize --- lap.optimize()


# Let's print the plan: which sites open, the reserves, the flows, the shortages, the cost

## Reference: the completed model

By hand --- no dictionaries, no loops, no `quicksum` --- so every term matches the math above.

In [ ]:
lap = gp.Model("lap")

# switches -- one per node, every node a candidate
xA = lap.addVar(vtype=GRB.BINARY, name="xA")
xB = lap.addVar(vtype=GRB.BINARY, name="xB")
xC = lap.addVar(vtype=GRB.BINARY, name="xC")
xD = lap.addVar(vtype=GRB.BINARY, name="xD")
xE = lap.addVar(vtype=GRB.BINARY, name="xE")

# reserves -- tons are divisible, so continuous (lower bound 0 by default)
rA = lap.addVar(name="rA")
rB = lap.addVar(name="rB")
rC = lap.addVar(name="rC")
rD = lap.addVar(name="rD")
rE = lap.addVar(name="rE")

# flows -- each road twice, once per direction
yAC = lap.addVar(name="yAC"); yCA = lap.addVar(name="yCA")
yBC = lap.addVar(name="yBC"); yCB = lap.addVar(name="yCB")
yCD = lap.addVar(name="yCD"); yDC = lap.addVar(name="yDC")
yCE = lap.addVar(name="yCE"); yEC = lap.addVar(name="yEC")
yDE = lap.addVar(name="yDE"); yED = lap.addVar(name="yED")

# shortfalls (capped by demand) and surpluses
qA = lap.addVar(ub=12, name="qA"); sA = lap.addVar(name="sA")
qB = lap.addVar(ub=20, name="qB"); sB = lap.addVar(name="sB")
qC = lap.addVar(ub=8,  name="qC"); sC = lap.addVar(name="sC")
qD = lap.addVar(ub=16, name="qD"); sD = lap.addVar(name="sD")
qE = lap.addVar(ub=24, name="qE"); sE = lap.addVar(name="sE")

# objective -- open + hold + ship + fail + waste
lap.setObjective(
    100 * (xA + xB + xC + xD + xE)
    + 10 * (rA + rB + rC + rD + rE)
    + 4.5 * (yAC + yCA) + 4.9 * (yBC + yCB) + 5.1 * (yCD + yDC)
    + 5.2 * (yCE + yEC) + 4.0 * (yDE + yED)
    + 60 * (qA + qB + qC + qD + qE)
    + (sA + sB + sC + sD + sE),
    GRB.MINIMIZE)

# conservation -- MCNF's row at every node: outflow - inflow = r - d + q - s
nodeA = lap.addConstr(yAC - yCA == rA - 12 + qA - sA, name="nodeA")
nodeB = lap.addConstr(yBC - yCB == rB - 20 + qB - sB, name="nodeB")
nodeC = lap.addConstr(yCA + yCB + yCD + yCE - yAC - yBC - yDC - yEC == rC - 8 + qC - sC, name="nodeC")
nodeD = lap.addConstr(yDC + yDE - yCD - yED == rD - 16 + qD - sD, name="nodeD")
nodeE = lap.addConstr(yEC + yED - yCE - yDE == rE - 24 + qE - sE, name="nodeE")

# budget, linking (M = B = 55), at most p = 2 facilities
lap.addConstr(rA + rB + rC + rD + rE <= 55, name="budget")
lap.addConstr(rA <= 55 * xA, name="linkA")
lap.addConstr(rB <= 55 * xB, name="linkB")
lap.addConstr(rC <= 55 * xC, name="linkC")
lap.addConstr(rD <= 55 * xD, name="linkD")
lap.addConstr(rE <= 55 * xE, name="linkE")
lap.addConstr(xA + xB + xC + xD + xE <= 2, name="cardinality")

lap.optimize()

print(f"\nopen: A={round(xA.X)} B={round(xB.X)} C={round(xC.X)} D={round(xD.X)} E={round(xE.X)}")
print(f"reserves: rB = {rB.X:g}, rE = {rE.X:g}")
print(f"flows: E->D = {yED.X:g}   shortages: qA = {qA.X:g}, qC = {qC.X:g}, qD = {qD.X:g}")
print(f"total cost = {lap.ObjVal:g}   (the game's do-nothing plan costs 4800)")

## Round 2 --- declare everything integer

Crates ship whole: flip $y, r, q, s$ from continuous to integer and re-optimize. Prediction before running: harder, easier, or no change?

In [ ]:
# flip the vtypes -- no other change
for v in [rA, rB, rC, rD, rE, yAC, yCA, yBC, yCB, yCD, yDC, yCE, yEC, yDE, yED,
          qA, qB, qC, qD, qE, sA, sB, sC, sD, sE]:
    v.VType = GRB.INTEGER

lap.optimize()

print(f"\nopen: B={round(xB.X)} E={round(xE.X)}   rB = {rB.X:g}, rE = {rE.X:g}, E->D = {yED.X:g}")
print(f"total cost = {lap.ObjVal:g}   -- same plan, same 2294")

**Same plan, same 2294 --- and the continuous solve was already integer.** Why: fix $x$, and the linking rows become plain bounds. What is left in $(r, y, q, s)$ is an MCNF on an augmented graph --- $r_j$ is an arc from a super-source (the budget row is its capacity $B$), $q_j$ and $s_j$ are shortfall and disposal arcs. Every column holds one $+1$ and one $-1$: **TU** by week 5's checklist ($M_1$ = all rows, $M_2 = \emptyset$). Integer data $\Rightarrow$ integer corners. Integrality on the flow side never binds; branch \& bound only ever has to work on $x$.

## Round 3 --- the contrast: relax $x$ instead

The hardness is not symmetric. Relax **everything** (in particular the switches) and watch what breaks.

In [ ]:
rel = lap.relax()   # drops every integrality declaration
rel.optimize()

xs = {n: rel.getVarByName(f"x{n}").X for n in "ABCDE"}
rs = {n: rel.getVarByName(f"r{n}").X for n in "ABCDE"}
print("\nx:", {n: round(v, 3) for n, v in xs.items() if v > 1e-6}, "  sum =", round(sum(xs.values()), 3))
print("r:", {n: round(v, 1) for n, v in rs.items() if v > 1e-6}, "  -- and zero shipping")
print(f"z_LP = {rel.ObjVal:g}   <   z_IP = 2294")
print("one facility's open cost, smeared over three sites -- the gap lives in x, not in y")

## The caveat --- integer data

The TU guarantee needs an integer right-hand side. Make node E's demand fractional, $d_E = 24.5$, and integer flows start to cost real money. ($q, s$ stay continuous --- shortfall in tons can be fractional; make them integer too and the model goes **infeasible**: try it.)

In [ ]:
# d_E = 24.5 -- edit the row and the shortfall cap
nodeE.RHS = -24.5
qE.UB = 24.5

# continuous first
for v in [rA, rB, rC, rD, rE, yAC, yCA, yBC, yCB, yCD, yDC, yCE, yEC, yDE, yED,
          qA, qB, qC, qD, qE, sA, sB, sC, sD, sE]:
    v.VType = GRB.CONTINUOUS
lap.optimize()
z_cont = lap.ObjVal
print(f"\ncontinuous flows: E->D = {yED.X:g}, cost = {z_cont:g}")

# now whole crates: y and r integer
for v in [rA, rB, rC, rD, rE, yAC, yCA, yBC, yCB, yCD, yDC, yCE, yEC, yDE, yED]:
    v.VType = GRB.INTEGER
lap.optimize()
print(f"integer flows:    E->D = {yED.X:g}, qE = {qE.X:g}, cost = {lap.ObjVal:g}")
print(f"\nintegrality now binds: {lap.ObjVal:g} > {z_cont:g}")

**Takeaway.** With integer data, declaring $y, q, s$ integer or continuous does not affect the LAP --- the flow side is TU once $x$ is fixed, so its corners are integer for free. Declare integrality only where it binds: that is why the letters split in week 5 --- $x$ integer, $y$ continuous.